# Grupo 6 · Spambase | Datos, exploración, correlaciones e interpretaciones (E2)

**Asignatura:** Tecnologías Emergentes · **Dataset:** UCI Spambase (ID 94) · **Semilla:** 42  
**Objetivo:** inspeccionar y validar el dataset, documentar las incidencias, visualizar variables y realizar la partición estratificada sin fuga de información.

**Trazabilidad:** las celdas imprimen resultados obtenidos al ejecutarse. No se presuponen errores, faltantes ni métricas antes de verificar el archivo. **No usar el conjunto de prueba para elegir modelos o umbrales.**

> Notebook basado en el archivo inicial del equipo. Corrección registrada: `StandartScaler` → `StandardScaler`. Las fechas y autores de ejecución deben completarse al correrlo en VS Code/Jupyter.

**Ampliación:** matriz de correlación de 57×57, tabla de 15 pares con mayor correlación y análisis equivalente solo sobre entrenamiento. No se realiza eliminación automática de variables.

In [ ]:
import sys, platform, json, hashlib
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)
np.random.seed(42)
BASE = Path.cwd()
DATOS = BASE / 'datos'
SALIDA = BASE / 'resultados_eda'
DATOS.mkdir(exist_ok=True, parents=True)
SALIDA.mkdir(exist_ok=True, parents=True)
bitacora = []
def registrar(etapa, observacion, decision, evidencia=''):
    bitacora.append({'fecha_utc': datetime.now(timezone.utc).isoformat(), 'etapa': etapa,
                    'observacion': str(observacion), 'decision': str(decision), 'evidencia': str(evidencia)})
print('Entorno:', sys.version.split()[0], '| Pandas:', pd.__version__, '| SO:', platform.platform())
registrar('01. Entorno', 'Importaciones correctas y entorno identificado', 'Se corrige StandartScaler por StandardScaler en el notebook inicial.')

## 1. Fuente, diccionario y carga
**Fuente oficial:** https://archive.ics.uci.edu/dataset/94/spambase  
La matriz contiene 57 características continuas y una etiqueta binaria (`spam`: 1, `no_spam`: 0). Las características del texto corresponden a estadísticas previamente extraídas; no equivalen a almacenar el correo original.

In [ ]:
WORDS = ['make','address','all','3d','our','over','remove','internet','order','mail','receive','will',
         'people','report','addresses','free','business','email','you','credit','your','font','000','money',
         'hp','hpl','george','650','lab','labs','telnet','857','data','415','85','technology','1999',
         'parts','pm','direct','cs','meeting','original','project','re','edu','table','conference']
CHARS = [';', '(', '[', '!', '$', '#']
FEATURES = [f'word_freq_{w}' for w in WORDS] + [f'char_freq_{ch}' for ch in CHARS] + [
    'capital_run_length_average','capital_run_length_longest','capital_run_length_total']
COLUMNAS = FEATURES + ['spam']
assert len(FEATURES) == 57 and len(set(FEATURES)) == 57
pd.DataFrame({'posicion': range(1,58),'variable': FEATURES}).to_csv(SALIDA/'diccionario_57_variables.csv',index=False)
print('Características:', len(FEATURES), '| columnas incluyendo objetivo:', len(COLUMNAS))

In [ ]:
# Si la descarga automática falla: descargar spambase.data del repositorio UCI y copiarlo en datos/spambase.data.
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data'
ruta = DATOS / 'spambase.data'
if not ruta.exists():
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        ruta.write_bytes(r.content)
        print('Archivo descargado desde UCI')
    except requests.RequestException as exc:
        raise FileNotFoundError('No se pudo descargar el dataset. Guarda manualmente spambase.data en datos/. Fuente: '+url) from exc
sha256 = hashlib.sha256(ruta.read_bytes()).hexdigest()
df = pd.read_csv(ruta, header=None, names=COLUMNAS)
if df.shape[1] != 58:
    raise ValueError(f'Se esperaban 58 columnas, llegaron {df.shape[1]}. Verifica el archivo.')
print('Dimensiones reales:', df.shape, '| SHA256:', sha256)
display(df.head())
registrar('02. Carga', f'Archivo {ruta.name}; forma {df.shape}; SHA256 {sha256}', 'Se conservó el archivo original sin transformarlo', str(ruta))

**Interpretación de la carga.** La cantidad de filas y columnas mostrada por el programa permite verificar si el archivo coincide con la estructura esperada (57 características y una etiqueta). El identificador SHA-256 permite comprobar que se trabajó con el mismo archivo. El hecho de que la carga termine sin errores no demuestra, por sí solo, que los datos sean de buena calidad.

## 2. Auditoría de calidad: tipos, ausentes, dominios y duplicados

In [ ]:
print('Tipos de columnas:')
display(df.dtypes.value_counts().rename('numero_columnas').to_frame())
print('Valores objetivo:', df['spam'].value_counts(dropna=False).to_dict())
print('Celdas nulas:', int(df.isna().sum().sum()))
reporte_calidad = pd.DataFrame({'variable':df.columns,'tipo':df.dtypes.astype(str).values,
                                'faltantes':df.isna().sum().values,'unicos':df.nunique(dropna=False).values})
display(reporte_calidad.head(12))
reporte_calidad.to_csv(SALIDA/'reporte_calidad_columnas.csv',index=False)
registrar('03. Faltantes',f'{int(df.isna().sum().sum())} celdas nulas',
         'No imputar por rutina; imputar solo si existen faltantes, mediante Pipeline ajustado con TRAIN.')

In [ ]:
from IPython.display import Markdown, display
faltantes = int(df.isna().sum().sum())
columnas_faltantes = int((df.isna().sum()>0).sum())
display(Markdown(f"""**Interpretación automática — datos faltantes.** Se analizaron **{len(df):,} registros** y **{len(df.columns)} columnas**. Se encontraron **{faltantes} celdas faltantes** distribuidas en **{columnas_faltantes} columnas**. """ + ("No se requiere imputación por ausencia de datos en el archivo observado; se conserva un imputador en la canalización por robustez." if faltantes==0 else "Es necesario investigar el origen de los faltantes; cualquier imputación debe ajustarse solo con entrenamiento.")))

In [ ]:
# Detectar inválidos por dominio; las frecuencias de palabras/caracteres son no negativas,
# y las tres métricas de mayúsculas tampoco pueden ser negativas.
X_num = df[FEATURES].apply(pd.to_numeric, errors='coerce')
no_numericos = int(X_num.isna().sum().sum() - df[FEATURES].isna().sum().sum())
negativos = int((X_num < 0).sum().sum())
obj_invalido = int((~df['spam'].isin([0,1])).sum())
duplicados_exactos = int(df.duplicated().sum())
duplicados_caracteristicas = int(df.duplicated(subset=FEATURES).sum())
print({'valores_no_numericos':no_numericos, 'valores_negativos':negativos,
       'etiquetas_invalidas':obj_invalido, 'filas_duplicadas_exactas':duplicados_exactos,
       'filas_repetidas_en_X':duplicados_caracteristicas})
if no_numericos or negativos or obj_invalido:
    raise ValueError('Existen registros de dominio inválido. Analizar con el equipo antes de corregir.')
registrar('04. Dominios', f'No numéricos: {no_numericos}; negativos: {negativos}; etiquetas inválidas: {obj_invalido}',
         'Detener ante valores inválidos en vez de corregirlos silenciosamente.')
registrar('05. Duplicados', f'Exactos: {duplicados_exactos}; mismas características: {duplicados_caracteristicas}',
         'No eliminar automáticamente: revisar riesgo de fuga entre particiones y procedencia.')

In [ ]:
display(Markdown(f"""**Interpretación automática — calidad.** Se detectaron **{no_numericos} valores no numéricos**, **{negativos} valores negativos**, **{obj_invalido} etiquetas inválidas**, **{duplicados_exactos} filas exactamente duplicadas** y **{duplicados_caracteristicas} registros con características repetidas**. Las repeticiones no se eliminan sin analizar su procedencia; se mantienen juntas en las particiones para prevenir que ejemplos idénticos aparezcan en entrenamiento y prueba. """))

## 3. Descripción y distribución de clases

Se muestran estadísticas descriptivas y el balance de etiquetas. Los gráficos se guardan en `resultados_eda/`.

In [ ]:
display(df[FEATURES].describe().T.round(3).head(20))
conteos = df['spam'].value_counts().reindex([0,1],fill_value=0)
fig,ax=plt.subplots(figsize=(6,4))
sns.barplot(x=['No spam (0)','Spam (1)'],y=conteos.values,ax=ax,color='#3977B9')
for i,v in enumerate(conteos.values): ax.text(i,v,str(v),ha='center',va='bottom')
ax.set(title='Distribución de la variable objetivo',ylabel='Cantidad de registros',xlabel='Clase')
fig.tight_layout();fig.savefig(SALIDA/'01_balance_clases.png',dpi=160);plt.show()
registrar('06. Clases', conteos.to_dict(), 'Se utilizará estratificación para conservar proporciones.')

In [ ]:
total=int(conteos.sum()); nspam=int(conteos.loc[1]); nleg=int(conteos.loc[0]); pct=100*nspam/total
display(Markdown(f"""**Interpretación automática — clases.** El conjunto cargado contiene **{nleg} correos no spam ({100-pct:.2f}%)** y **{nspam} correos spam ({pct:.2f}%)**. Una clase representa **{max(pct,100-pct):.2f}%** de los registros; por ello, la exactitud sola no basta para valorar el clasificador. Se reportarán precisión, recall, F1 y PR-AUC, además de una línea base. """))

## 4. Variables, correlaciones y valores atípicos

In [ ]:
seleccion = ['word_freq_free','word_freq_you','word_freq_george','char_freq_!',
             'char_freq_$','capital_run_length_total']
fig,axes=plt.subplots(2,3,figsize=(13,7))
for ax,col in zip(axes.flat,seleccion):
    sns.boxplot(data=df,x='spam',y=col,ax=ax,showfliers=True)
    ax.set_title(col);ax.set_xlabel('0 = no spam, 1 = spam')
fig.tight_layout();fig.savefig(SALIDA/'02_boxplots_por_clase.png',dpi=160);plt.show()
# Un valor extremo NO se elimina automáticamente: podría contener señal útil sobre spam.
q1=X_num.quantile(.25);q3=X_num.quantile(.75);iqr=q3-q1
marcas=((X_num.lt(q1-1.5*iqr)) | (X_num.gt(q3+1.5*iqr)))
top=marcas.sum().sort_values(ascending=False).head(12)
display(top.rename('valores_fuera_de_bigotes_IQR').to_frame())
registrar('07. Atípicos', f'Detección descriptiva por IQR. Top: {top.to_dict()}',
         'Conservar inicialmente; no representan automáticamente errores de captura.')

In [ ]:
# Mostrar asociaciones descriptivas sin usar prueba final para seleccionar características.
correlacion = df[FEATURES+['spam']].corr(numeric_only=True)['spam'].drop('spam').sort_values()
top_corr = pd.concat([correlacion.head(8),correlacion.tail(8)]).sort_values()
fig,ax=plt.subplots(figsize=(9,6))
top_corr.plot(kind='barh',ax=ax,color='#3977B9')
ax.set(title='Correlación exploratoria con etiqueta spam',xlabel='Correlación de Pearson')
fig.tight_layout();fig.savefig(SALIDA/'03_correlaciones_objetivo.png',dpi=160);plt.show()
registrar('08. Correlaciones', 'Asociaciones descriptivas calculadas para EDA',
         'No seleccionar variables con información de validación/prueba en un flujo de entrenamiento.')

**Interpretación — correlación con la etiqueta.** El signo de Pearson indica dirección de asociación lineal, no causalidad. Un coeficiente positivo indica que valores más altos de la característica suelen asociarse con `spam=1`; uno negativo, con `spam=0`. Esta exploración general ayuda a conocer el dataset, pero **no debe utilizarse para seleccionar variables**, ya que incluye registros que luego pertenecerán a validación y prueba. La selección para entrenamiento se hará exclusivamente con `X_train`.

### 4.1. Matriz completa de correlación de las 57 características

La siguiente matriz de Pearson permite **visualizar posibles redundancias** entre las 57 variables predictoras. Rojo = relación positiva; azul = negativa; la diagonal vale 1. **Es un análisis descriptivo**, no una justificación para eliminar columnas automáticamente. El gráfico completo usa todos los registros con fines de descripción; cualquier **selección de características para modelado se calculará exclusivamente con entrenamiento** (véase 4.3).

In [ ]:
# Correlaciones entre características (sin incluir la etiqueta spam).
matriz_corr_57 = df[FEATURES].corr(method='pearson')
fig, ax = plt.subplots(figsize=(19, 16))
sns.heatmap(matriz_corr_57, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, annot=False, linewidths=0.15, ax=ax,
            cbar_kws={'label': 'Coeficiente de Pearson'})
ax.set_title('Spambase: correlación de las 57 características', fontsize=16, pad=14)
ax.tick_params(axis='x', labelrotation=90, labelsize=6)
ax.tick_params(axis='y', labelsize=6)
fig.tight_layout()
fig.savefig(SALIDA / '04_matriz_correlacion_57.png', dpi=180, bbox_inches='tight')
plt.show()
registrar('08a. Matriz completa', 'Se calcularon 57 × 57 correlaciones de Pearson (solo atributos)',
         'Únicamente análisis descriptivo; no eliminar variables automáticamente.',
         'resultados_eda/04_matriz_correlacion_57.png')

**Interpretación — mapa de calor de 57 × 57.** En la diagonal aparece 1, porque cada característica se correlaciona consigo misma. Bloques de color intenso fuera de la diagonal pueden sugerir redundancia, pero no prueban que debamos eliminar columnas. Como la gráfica utiliza el conjunto completo con fines descriptivos, las decisiones de reducción deberán basarse únicamente en entrenamiento y compararse mediante validación.

### 4.2. Tabla de los 15 pares más correlacionados

Se muestran los pares ordenados por su **correlación absoluta**, conservando el signo del coeficiente. Un valor cercano a +1 o −1 indica relación lineal fuerte, pero no necesariamente redundancia predictiva.

In [ ]:
# Triángulo superior: evita diagonal y parejas repetidas.
mascara_superior = np.triu(np.ones(matriz_corr_57.shape, dtype=bool), k=1)
pares_corr = matriz_corr_57.where(mascara_superior).stack().rename('correlacion_pearson')
tabla_pares = (pares_corr.reset_index()
               .rename(columns={'level_0':'variable_1', 'level_1':'variable_2'}))
tabla_pares['correlacion_absoluta'] = tabla_pares['correlacion_pearson'].abs()
tabla_pares = tabla_pares.sort_values('correlacion_absoluta', ascending=False)
top15 = tabla_pares.head(15).reset_index(drop=True)
display(top15.round(4))
top15.to_csv(SALIDA / '05_top15_correlaciones.csv', index=False)
print('Pares con |r| >= 0.85 (umbral descriptivo):',
      int((tabla_pares['correlacion_absoluta'] >= .85).sum()))
registrar('08b. Pares correlacionados',
         f'Se listaron los 15 pares con mayor |r|. Pares >= 0.85: {int((tabla_pares["correlacion_absoluta"]>=.85).sum())}',
         'Solo candidatos a analizar; no se borraron características.',
         'resultados_eda/05_top15_correlaciones.csv')

In [ ]:
if len(top15):
    fila=top15.iloc[0]
    fuertes=int((tabla_pares['correlacion_absoluta']>=.85).sum())
    sentido='positiva' if fila['correlacion_pearson']>0 else 'negativa'
    display(Markdown(f"""**Interpretación automática — pares de variables.** Entre los pares evaluados, el más correlacionado es **`{fila['variable_1']}` / `{fila['variable_2']}`**, con **r = {fila['correlacion_pearson']:.3f}** (relación {sentido}). Se observaron **{fuertes} pares con |r| ≥ 0.85**. Este resultado justifica **probar**, pero no imponer, una reducción de dimensionalidad; la comparación debe hacerse sobre validación."""))
else:
    display(Markdown('**Interpretación:** no se pudieron calcular correlaciones entre pares con los datos disponibles.'))

## 5. Estrategia de calidad y particiones

**No se borra nada automáticamente.** Se mantiene el dataset original y se detecta si existen filas con idénticas características. Estas filas requieren atención: se agrupan por vector de características para evitar que una misma observación exacta aparezca a la vez en entrenamiento y validación/prueba. Si aparecen etiquetas contradictorias para las mismas características, se debe detener el proceso y decidir con evidencia antes de continuar.

**Proporción:** train 60 %, validación 20 %, prueba 20 %. **La prueba queda reservada**. Los índices se exportan para que los integrantes 2 y 3 utilicen exactamente la misma partición.

In [ ]:
# Evitar fugas por vectores idénticos: asignación de grupos completos a una partición.
# Cada vector idéntico recibe un identificador. Se exige etiqueta consistente por grupo.
from sklearn.model_selection import StratifiedGroupKFold
huellas = pd.util.hash_pandas_object(df[FEATURES],index=False).to_numpy()
contradicciones = int((df.assign(grupo=huellas).groupby('grupo')['spam'].nunique()>1).sum())
print('Grupos de características con etiquetas contradictorias:', contradicciones)
if contradicciones:
    print('ADVERTENCIA: revisar las etiquetas contradictorias; se conservan agrupadas para evitar fuga.')
    registrar('09a. Conflictos', f'{contradicciones} grupos con etiquetas contradictorias', 'No se eliminaron ni corrigieron etiquetas sin evidencia; se mantienen en un mismo grupo de partición.')
# División estratificada por grupos: prueba ~20 %, validación ~20 % del total.
y=df['spam'].astype(int).to_numpy()
sgkf=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42)
idx_dev,idx_test=next(sgkf.split(df[FEATURES],y,groups=huellas))
resto=df.iloc[idx_dev]; hue_resto=huellas[idx_dev]; y_resto=y[idx_dev]
sgkf2=StratifiedGroupKFold(n_splits=4,shuffle=True,random_state=43)
local_train,local_val=next(sgkf2.split(resto[FEATURES],y_resto,groups=hue_resto))
idx_train=idx_dev[local_train];idx_val=idx_dev[local_val]
partes={'train':idx_train,'val':idx_val,'test':idx_test}
assert sum(len(v) for v in partes.values()) == len(df)
assert len(set(huellas[idx_train])&set(huellas[idx_test]))==0
assert len(set(huellas[idx_val])&set(huellas[idx_test]))==0
assert len(set(huellas[idx_train])&set(huellas[idx_val]))==0
resumen=[]
for nombre,indices in partes.items():
    resumen.append({'particion':nombre,'filas':len(indices),'porcentaje':round(len(indices)/len(df)*100,2),
                    'proporcion_spam':round(float(y[indices].mean()),4)})
    pd.DataFrame({'indice_original':indices}).to_csv(DATOS/f'indices_{nombre}.csv',index=False)
display(pd.DataFrame(resumen))
registrar('09. Particiones',resumen, 'StratifiedGroupKFold para respetar etiquetas y evitar grupos idénticos entre particiones.')

In [ ]:
part_df=pd.DataFrame(resumen)
texto='; '.join(f"{r.particion}: {int(r.filas)} registros ({r.porcentaje:.2f}%, spam {100*r.proporcion_spam:.2f}%)" for r in part_df.itertuples())
display(Markdown(f"**Interpretación automática — particiones.** Se generaron particiones por grupos de vectores idénticos: {texto}. Las aserciones comprueban que ningún grupo idéntico figure en dos particiones. La división por grupos puede producir porcentajes algo diferentes de una proporción 60/20/20 exacta. El conjunto de prueba permanecerá reservado hasta la evaluación definitiva."))

## 6. Imputación, codificación y escalado sin fuga

Spambase contiene **57 características numéricas**, así que **no es necesario OneHotEncoder ni LabelEncoder para la matriz predictora**. La variable objetivo ya es binaria (0/1). Se conserva un Pipeline con imputación y escalado, ajustado únicamente con entrenamiento. El preprocesamiento definitivo debe viajar junto con el modelo: el integrante 2 puede usar `Pipeline` o guardar el preprocesador ajustado.

In [ ]:
from sklearn import set_config
X_train=df.iloc[idx_train][FEATURES]
X_val=df.iloc[idx_val][FEATURES]
X_test=df.iloc[idx_test][FEATURES]
y_train=y[idx_train]; y_val=y[idx_val]; y_test=y[idx_test]
preprocesador = Pipeline([('imputador',SimpleImputer(strategy='median')),('escalador',StandardScaler())])
X_train_p=preprocesador.fit_transform(X_train)   # FIT solo train
X_val_p=preprocesador.transform(X_val)          # sin FIT en validación
X_test_p=preprocesador.transform(X_test)        # sin FIT en prueba
print('Formas:',X_train_p.shape,X_val_p.shape,X_test_p.shape)
print('Promedio de entrenamiento tras escalar (primeras 5 variables):',np.round(X_train_p.mean(axis=0)[:5],5))
import joblib
joblib.dump(preprocesador, DATOS/'preprocesador_referencia_train.joblib')
registrar('10. Preprocesamiento',f'Train {X_train_p.shape}, val {X_val_p.shape}, test {X_test_p.shape}',
         'Imputador mediana y StandardScaler ajustados solo sobre train; no aplicar fit en val/test.')

**Interpretación — preprocesamiento.** El método `fit_transform` calcula mediana y estadísticas de escalado solamente con entrenamiento. En validación y prueba se usa `transform` para reutilizar esas mismas estadísticas, sin aprender de los datos reservados. El escalado no garantiza por sí mismo un mejor modelo: la comparación corresponderá a E3 y E4.

### 4.3. Correlaciones calculadas solamente con entrenamiento (sin fuga)

Para formular **hipótesis de selección** sin utilizar validación ni prueba, recalculamos las correlaciones usando únicamente `X_train`. Este resultado no cambia los 57 atributos de la API ni elimina variables. Si se evalúa selección o PCA, deben colocarse dentro de un `Pipeline` ajustado en entrenamiento y compararse con el modelo completo utilizando validación.

In [ ]:
corr_train = X_train.corr(method='pearson')
pares_train = corr_train.where(np.triu(np.ones(corr_train.shape, dtype=bool), k=1)).stack()
resumen_pares_train = (pares_train.rename('pearson').reset_index()
                       .rename(columns={'level_0': 'variable_1', 'level_1': 'variable_2'}))
resumen_pares_train['abs_pearson'] = resumen_pares_train['pearson'].abs()
resumen_pares_train = resumen_pares_train.sort_values('abs_pearson', ascending=False)
display(resumen_pares_train.head(15).round(4))
resumen_pares_train.head(15).to_csv(SALIDA / '06_top15_correlaciones_SOLO_TRAIN.csv', index=False)
registrar('10a. Correlaciones train', 'Pares correlacionados recalculados únicamente con X_train',
         'Cualquier reducción de dimensionalidad deberá ajustarse en train y validarse antes de aprobarla.',
         'resultados_eda/06_top15_correlaciones_SOLO_TRAIN.csv')

In [ ]:
if len(resumen_pares_train):
    mayor=resumen_pares_train.iloc[0]
    display(Markdown(f"**Interpretación automática — correlaciones sin fuga.** En entrenamiento, el par con mayor correlación absoluta es **`{mayor['variable_1']}` y `{mayor['variable_2']}`** (r={mayor['pearson']:.3f}). Cualquier selección de características o PCA deberá ajustarse exclusivamente con estas observaciones de entrenamiento y validarse frente al modelo con las 57 variables originales. La matriz descriptiva del dataset completo no determina qué atributos se eliminan."))

## 7. Comprobaciones finales y exportación reproducible

Se exportan los índices, diccionario, bitácora y resumen de calidad. **Los índices de prueba no deben usarse para seleccionar umbral o arquitectura**. Idealmente, `entrenar.py` reutilizará los índices y construirá el Pipeline completo desde datos crudos. No subir información sensible; Spambase es un dataset tabular público.

In [ ]:
resumen_ejecucion = {
    'fecha_utc':datetime.now(timezone.utc).isoformat(), 'fuente':'UCI Spambase ID 94',
    'sha256_datos_originales':sha256, 'filas':int(len(df)),'features':len(FEATURES),
    'celdas_faltantes':int(df.isna().sum().sum()),'duplicados_exactos':duplicados_exactos,
    'duplicados_X':duplicados_caracteristicas,'grupos_contradictorios':contradicciones,
    'particiones':resumen, 'semillas':{'outer':42,'inner':43},
    'regla':'No se usó test para decidir limpieza, modelo o umbral; partición agrupada por vector exacto.'}
(SALIDA/'resumen_ejecucion.json').write_text(json.dumps(resumen_ejecucion,ensure_ascii=False,indent=2),encoding='utf-8')
pd.DataFrame(bitacora).to_csv(SALIDA/'bitacora_observaciones.csv',index=False)
display(pd.DataFrame(bitacora)[['etapa','observacion','decision']])
print('Archivos generados en:', SALIDA.resolve(), 'y', DATOS.resolve())

In [ ]:
interpretacion_final = f"""# Resumen interpretativo de E2 — Spambase Grupo 6

- Registros analizados: {len(df)}; características predictoras: {len(FEATURES)}.
- Valores faltantes: {faltantes}; duplicados exactos: {duplicados_exactos}; vectores repetidos: {duplicados_caracteristicas}.
- Distribución: no spam {nleg} ({100-pct:.2f}%) y spam {nspam} ({pct:.2f}%).
- Pares con correlación absoluta mayor o igual a 0.85 (EDA): {int((tabla_pares['correlacion_absoluta']>=.85).sum())}.
- Particiones: {texto}.
- Decisión: no eliminar variables ni observaciones sin evidencia. Entrenar un modelo inicial con 57 atributos y evaluar posteriormente si la selección de características o PCA mejora el desempeño en validación.
- Restricción: la prueba final no debe usarse para elegir modelos ni transformaciones.

Estas interpretaciones provienen de la ejecución actual del notebook; no son métricas de modelos entrenados.
"""
(SALIDA/'interpretaciones_para_informe.md').write_text(interpretacion_final, encoding='utf-8')
display(Markdown(interpretacion_final))
print('Interpretaciones guardadas en resultados_eda/interpretaciones_para_informe.md')

## 8. Conclusiones y decisiones para el informe

Al ejecutar el notebook, redactar las conclusiones **a partir de las salidas reales**:

1. Reportar número de filas, clases y faltantes observados; adjuntar `resumen_ejecucion.json`.
2. Indicar si se detectaron duplicados exactos y por características. Explicar que se optó por particionar por grupos idénticos para reducir riesgo de fuga, en lugar de borrarlos sin justificar.
3. Describir valores extremos como *observaciones atípicas* y explicar por qué no se eliminaron automáticamente.
4. Precisar que no se requirió codificación categórica y que el escalado se ajustó solo con entrenamiento.
5. Documentar diferencias de proporciones reales respecto al objetivo 60/20/20, porque la división por grupos puede ser aproximada.
6. **Limitación importante:** exploraciones globales de correlación y etiquetas son descriptivas y no deben utilizarse para seleccionar características antes de evaluar modelos. Para selección de características, analizar exclusivamente entrenamiento.

**Declaración de uso de IA:** indicar las herramientas utilizadas para redactar/depurar el notebook; verificar personalmente celdas, gráficos y conclusiones. La bitácora genera observaciones automáticamente, pero cada hallazgo debe revisarse por el equipo.

**Fuente:** UCI Machine Learning Repository — Spambase, https://archive.ics.uci.edu/dataset/94/spambase.